# Catálogo de fundos multimercados (universo do estudo)

Universo fixo de 27 fundos multimercados (planilha `data/catalogo_fundos_multimercados.xlsx`, formato BTG), com classificações, taxas e métricas de performance já calculadas pela distribuidora. Este notebook explora o catálogo e, na seção final, cruza os CNPJs com os dados diários da CVM para recalcular as métricas de forma independente.

In [ ]:
import sys
sys.path.insert(0, "../src")

from fundos.catalogo import carregar_catalogo, ranking, resumo_por_gestora, cnpjs
import pandas as pd

catalogo = carregar_catalogo()
catalogo[["fundo", "gestora", "classificacao_anbima", "patrimonio_liquido", "retorno_nominal_12m", "sharpe_12m"]]

## Rankings (12 meses)

In [ ]:
print("Melhor Sharpe 12M:")
display(ranking(catalogo, "sharpe_12m", top_n=10))

print("Maior retorno %CDI 12M:")
display(ranking(catalogo, "retorno_pct_cdi_12m", top_n=10))

print("Menor volatilidade 12M:")
display(ranking(catalogo, "volatilidade_12m", top_n=10, ascendente=True))

## Visão por gestora

In [ ]:
resumo_por_gestora(catalogo)

## Classificação Anbima e taxas

In [ ]:
catalogo.groupby("classificacao_anbima").agg(
    n_fundos=("fundo", "count"),
    taxa_administracao_media=("taxa_administracao", "mean"),
    taxa_performance_media=("taxa_performance", "mean"),
    retorno_12m_medio=("retorno_nominal_12m", "mean"),
)

## Cruzando com a CVM: séries diárias e verificação independente

As métricas acima vêm prontas da planilha. Para analisar a série histórica completa (evolução da cota dia a dia, drawdown, correlação entre fundos) usamos o CNPJ de cada fundo para baixar o informe diário da CVM e recalcular tudo com `fundos.metrics`.

> Requer acesso à internet (`dados.cvm.gov.br` e `api.bcb.gov.br`).

In [ ]:
from fundos.cvm import fetch_informe_periodo, cotas_do_fundo
from fundos.benchmarks import cdi_diario
from fundos.metrics import resumo, retornos_diarios, matriz_correlacao

# Top 5 por Sharpe, para não baixar o universo inteiro
top5 = ranking(catalogo, "sharpe_12m", top_n=5)
top5_cnpjs = catalogo[catalogo["fundo"].isin(top5["fundo"])][["fundo", "cnpj"]]
top5_cnpjs

In [ ]:
informe = fetch_informe_periodo("2025-08", "2026-07")  # últimos 12 meses

series_por_fundo = {
    row["fundo"]: cotas_do_fundo(informe, row["cnpj"]) for _, row in top5_cnpjs.iterrows()
}

cdi = cdi_diario("2025-08-01", "2026-07-31")

for nome, cotas in series_por_fundo.items():
    print(nome, resumo(cotas, retornos_livre_risco=cdi))

In [ ]:
retornos_por_fundo = {nome: retornos_diarios(cotas) for nome, cotas in series_por_fundo.items()}
matriz_correlacao(retornos_por_fundo)